# Step 1: RDD & DataFrame Basics

## Learning Objectives
1. Create SparkSession and connect to cluster
2. RDD concepts and basic operations (Transformation vs Action)
3. Create and operate on DataFrames
4. Understand Lazy Evaluation
5. Performance comparison: RDD vs DataFrame

---
## 1. Creating SparkSession

SparkSession, introduced in Spark 2.0, is the **entry point to all Spark functionality**.
- Consolidates the previous SparkContext, SQLContext, and HiveContext
- One SparkSession per JVM (or multiple)

In [1]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("Step1-RDD-DataFrame-Basics") \
    .master("spark://spark-master:7077") \
    .config("spark.executor.memory", "1g") \
    .config("spark.executor.cores", "1") \
    .getOrCreate()

sc = spark.sparkContext
print(f"Spark version: {spark.version}")
print(f"Master: {sc.master}")
print(f"App Name: {sc.appName}")
print(f"\n✅ Spark UI: http://localhost:4040")
print(f"✅ Master UI: http://localhost:8080")

Spark version: 3.5.0
Master: spark://spark-master:7077
App Name: Step1-RDD-DataFrame-Basics

✅ Spark UI: http://localhost:4040
✅ Master UI: http://localhost:8080


---
## 2. RDD (Resilient Distributed Dataset)

RDD is the most fundamental data structure in Spark.

**Key Characteristics:**
- **Resilient**: Auto-recovery on failure (lineage-based)
- **Distributed**: Data stored across cluster nodes
- **Dataset**: Immutable collection of data

### Transformation vs Action
| Type | Transformation | Action |
|------|---------------|--------|
| When | **Lazy** - only builds execution plan | **Eager** - performs actual computation |
| Returns | New RDD | Value (Python object) |
| Examples | map, filter, flatMap, groupByKey | collect, count, first, take, reduce |

### 2.1 Creating RDDs

In [2]:
# Method 1: Create from Python list (parallelize)
numbers = list(range(1, 101))
rdd = sc.parallelize(numbers, numSlices=4)  # distribute into 4 partitions

print(f"Type: {type(rdd)}")
print(f"Partitions: {rdd.getNumPartitions()}")
print(f"First 5 elements: {rdd.take(5)}")

Type: <class 'pyspark.rdd.RDD'>
Partitions: 4
First 5 elements: [1, 2, 3, 4, 5]


In [3]:
# Method 2: Create from text data
text_rdd = sc.parallelize([
    "Apache Spark is a unified analytics engine",
    "Spark provides an interface for programming clusters",
    "Spark supports Java Scala Python and R",
    "Spark runs on Hadoop YARN Kubernetes and standalone",
    "Spark is fast and general purpose"
])

print(f"Line count: {text_rdd.count()}")

Line count: 5


### 2.2 Transformation Practice

Transformations **return a new RDD**, and actual computation is deferred until an Action is called.

In [4]:
# map: apply function to each element
squared = rdd.map(lambda x: x ** 2)
print(f"map result (first 5): {squared.take(5)}")

# filter: keep only elements matching condition
evens = rdd.filter(lambda x: x % 2 == 0)
print(f"Even count: {evens.count()}")

# flatMap: map + flatten
words = text_rdd.flatMap(lambda line: line.split(" "))
print(f"Total words: {words.count()}")
print(f"First 10 words: {words.take(10)}")

map result (first 5): [1, 4, 9, 16, 25]
Even count: 50
Total words: 35
First 10 words: ['Apache', 'Spark', 'is', 'a', 'unified', 'analytics', 'engine', 'Spark', 'provides', 'an']


In [5]:
# 🔥 Word Count - The Hello World of MapReduce
word_counts = (
    text_rdd
    .flatMap(lambda line: line.lower().split(" "))  # split into words
    .map(lambda word: (word, 1))                     # (word, 1) pairs
    .reduceByKey(lambda a, b: a + b)                 # sum by key
    .sortBy(lambda x: x[1], ascending=False)         # sort by frequency
)

print("=== Word Count Results ===")
for word, count in word_counts.collect():
    print(f"  {word:20s} → {count}")

=== Word Count Results ===
  spark                → 5
  and                  → 3
  is                   → 2
  apache               → 1
  interface            → 1
  java                 → 1
  python               → 1
  runs                 → 1
  hadoop               → 1
  fast                 → 1
  an                   → 1
  for                  → 1
  analytics            → 1
  supports             → 1
  general              → 1
  on                   → 1
  kubernetes           → 1
  standalone           → 1
  scala                → 1
  r                    → 1
  a                    → 1
  unified              → 1
  engine               → 1
  provides             → 1
  programming          → 1
  yarn                 → 1
  purpose              → 1
  clusters             → 1


### 2.3 Verifying Lazy Evaluation

Transformations do not execute immediately. Only the execution plan is recorded, and computation runs all at once when an Action is called.

**Why Lazy?**
- Avoids storing unnecessary intermediate results
- Allows the Catalyst Optimizer to optimize the entire pipeline

In [6]:
import time

# Build transformation chain - no computation happens here
start = time.time()
lazy_rdd = (
    sc.parallelize(range(1, 10_000_001))
    .filter(lambda x: x % 3 == 0)
    .map(lambda x: x * x)
    .filter(lambda x: x % 7 == 0)
)
transform_time = time.time() - start
print(f"Transformation chain built: {transform_time:.4f}s (nearly instant!)")

# Call Action - actual computation happens here
start = time.time()
result_count = lazy_rdd.count()
action_time = time.time() - start
print(f"Action (count) executed: {action_time:.4f}s")
print(f"Result: {result_count:,} elements")
print(f"\n💡 Transformation is {action_time/max(transform_time, 0.0001):.0f}x faster → no actual computation")

Transformation chain built: 0.0013s (nearly instant!)
Action (count) executed: 0.4570s
Result: 476,190 elements

💡 Transformation is 343x faster → no actual computation


---
## 3. DataFrame

A DataFrame is a **distributed data table organized into named columns**.

**Advantages over RDD:**
- Schema-aware, suitable for structured data processing
- Automatically optimized by Catalyst Optimizer
- Memory-efficient via Tungsten engine
- Supports SQL queries

**In practice, DataFrame/Dataset is almost always preferred.**

### 3.1 Creating DataFrames

In [7]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, FloatType
from pyspark.sql import Row

# Method 1: From list + schema definition
schema = StructType([
    StructField("name", StringType(), False),
    StructField("department", StringType(), True),
    StructField("salary", IntegerType(), True),
    StructField("experience_years", IntegerType(), True)
])

data = [
    ("Alice", "Engineering", 95000, 5),
    ("Bob", "Engineering", 82000, 3),
    ("Charlie", "Marketing", 75000, 7),
    ("Diana", "Engineering", 110000, 8),
    ("Eve", "Marketing", 68000, 2),
    ("Frank", "Sales", 72000, 4),
    ("Grace", "Sales", 85000, 6),
    ("Heidi", "Engineering", 98000, 5),
    ("Ivan", "Marketing", 71000, 3),
    ("Judy", "Sales", 90000, 9),
]

df = spark.createDataFrame(data, schema)
df.show()
df.printSchema()

+-------+-----------+------+----------------+
|   name| department|salary|experience_years|
+-------+-----------+------+----------------+
|  Alice|Engineering| 95000|               5|
|    Bob|Engineering| 82000|               3|
|Charlie|  Marketing| 75000|               7|
|  Diana|Engineering|110000|               8|
|    Eve|  Marketing| 68000|               2|
|  Frank|      Sales| 72000|               4|
|  Grace|      Sales| 85000|               6|
|  Heidi|Engineering| 98000|               5|
|   Ivan|  Marketing| 71000|               3|
|   Judy|      Sales| 90000|               9|
+-------+-----------+------+----------------+

root
 |-- name: string (nullable = false)
 |-- department: string (nullable = true)
 |-- salary: integer (nullable = true)
 |-- experience_years: integer (nullable = true)



In [8]:
# Method 2: Using Row objects
rows = [
    Row(product="Laptop", price=1200.0, quantity=50),
    Row(product="Phone", price=800.0, quantity=150),
    Row(product="Tablet", price=500.0, quantity=80),
]
products_df = spark.createDataFrame(rows)
products_df.show()

+-------+------+--------+
|product| price|quantity|
+-------+------+--------+
| Laptop|1200.0|      50|
|  Phone| 800.0|     150|
| Tablet| 500.0|      80|
+-------+------+--------+



### 3.2 Basic DataFrame Operations

In [9]:
from pyspark.sql import functions as F

# select: choose columns
print("=== select ===")
df.select("name", "salary").show(5)

# filter (= where): conditional filtering
print("=== filter: salary > 80000 ===")
df.filter(F.col("salary") > 80000).show()

# withColumn: add/transform column
print("=== withColumn: convert salary to KRW 10K units ===")
df.withColumn("salary_krw_10k", F.col("salary") * 13.5 / 10000) \
  .select("name", "salary", "salary_krw_10k") \
  .show()

=== select ===
+-------+------+
|   name|salary|
+-------+------+
|  Alice| 95000|
|    Bob| 82000|
|Charlie| 75000|
|  Diana|110000|
|    Eve| 68000|
+-------+------+
only showing top 5 rows

=== filter: salary > 80000 ===
+-----+-----------+------+----------------+
| name| department|salary|experience_years|
+-----+-----------+------+----------------+
|Alice|Engineering| 95000|               5|
|  Bob|Engineering| 82000|               3|
|Diana|Engineering|110000|               8|
|Grace|      Sales| 85000|               6|
|Heidi|Engineering| 98000|               5|
| Judy|      Sales| 90000|               9|
+-----+-----------+------+----------------+

=== withColumn: convert salary to KRW 10K units ===
+-------+------+--------------+
|   name|salary|salary_krw_10k|
+-------+------+--------------+
|  Alice| 95000|        128.25|
|    Bob| 82000|         110.7|
|Charlie| 75000|        101.25|
|  Diana|110000|         148.5|
|    Eve| 68000|          91.8|
|  Frank| 72000|          9

In [10]:
# groupBy + agg: group aggregation
print("=== Department Statistics ===")
df.groupBy("department").agg(
    F.count("*").alias("headcount"),
    F.avg("salary").cast("int").alias("avg_salary"),
    F.max("salary").alias("max_salary"),
    F.min("salary").alias("min_salary"),
    F.avg("experience_years").alias("avg_experience")
).show()

# orderBy: sorting
print("=== Sorted by Salary ===")
df.orderBy(F.col("salary").desc()).show(5)

=== Department Statistics ===
+-----------+---------+----------+----------+----------+-----------------+
| department|headcount|avg_salary|max_salary|min_salary|   avg_experience|
+-----------+---------+----------+----------+----------+-----------------+
|Engineering|        4|     96250|    110000|     82000|             5.25|
|  Marketing|        3|     71333|     75000|     68000|              4.0|
|      Sales|        3|     82333|     90000|     72000|6.333333333333333|
+-----------+---------+----------+----------+----------+-----------------+

=== Sorted by Salary ===
+-----+-----------+------+----------------+
| name| department|salary|experience_years|
+-----+-----------+------+----------------+
|Diana|Engineering|110000|               8|
|Heidi|Engineering| 98000|               5|
|Alice|Engineering| 95000|               5|
| Judy|      Sales| 90000|               9|
|Grace|      Sales| 85000|               6|
+-----+-----------+------+----------------+
only showing top 5 rows

### 3.3 Spark SQL

Registering a DataFrame as a temporary view enables SQL queries.

In [11]:
# Register as temp view
df.createOrReplaceTempView("employees")

# Run SQL query
result = spark.sql("""
    SELECT 
        department,
        COUNT(*) as cnt,
        ROUND(AVG(salary), 0) as avg_salary,
        MAX(salary) - MIN(salary) as salary_range
    FROM employees
    GROUP BY department
    ORDER BY avg_salary DESC
""")

result.show()

+-----------+---+----------+------------+
| department|cnt|avg_salary|salary_range|
+-----------+---+----------+------------+
|Engineering|  4|   96250.0|       28000|
|      Sales|  3|   82333.0|       18000|
|  Marketing|  3|   71333.0|        7000|
+-----------+---+----------+------------+



In [12]:
# Window function example: salary ranking within department
rank_result = spark.sql("""
    SELECT 
        name,
        department,
        salary,
        RANK() OVER (PARTITION BY department ORDER BY salary DESC) as dept_rank,
        salary - AVG(salary) OVER (PARTITION BY department) as diff_from_avg
    FROM employees
    ORDER BY department, dept_rank
""")

rank_result.show()

+-------+-----------+------+---------+-------------------+
|   name| department|salary|dept_rank|      diff_from_avg|
+-------+-----------+------+---------+-------------------+
|  Diana|Engineering|110000|        1|            13750.0|
|  Heidi|Engineering| 98000|        2|             1750.0|
|  Alice|Engineering| 95000|        3|            -1250.0|
|    Bob|Engineering| 82000|        4|           -14250.0|
|Charlie|  Marketing| 75000|        1| 3666.6666666666715|
|   Ivan|  Marketing| 71000|        2| -333.3333333333285|
|    Eve|  Marketing| 68000|        3|-3333.3333333333285|
|   Judy|      Sales| 90000|        1| 7666.6666666666715|
|  Grace|      Sales| 85000|        2| 2666.6666666666715|
|  Frank|      Sales| 72000|        3|-10333.333333333328|
+-------+-----------+------+---------+-------------------+



---
## 4. Reading Execution Plans

`explain()` lets you see how Spark will execute a query.

This is what the **Catalyst Optimizer** does:
1. Unresolved Logical Plan → Logical Plan (Analysis)
2. Logical Plan → Optimized Logical Plan (Optimization)
3. Optimized Logical Plan → Physical Plan (Planning)
4. Physical Plan → RDD (Code Generation)

In [13]:
# Execution plan for a simple query
print("=== Basic Execution Plan ===")
df.filter(F.col("salary") > 80000) \
  .select("name", "department", "salary") \
  .explain()

print("\n=== Full Execution Plan (extended) ===")
df.filter(F.col("salary") > 80000) \
  .select("name", "department", "salary") \
  .explain(True)

=== Basic Execution Plan ===
== Physical Plan ==
*(1) Project [name#0, department#1, salary#2]
+- *(1) Filter (isnotnull(salary#2) AND (salary#2 > 80000))
   +- *(1) Scan ExistingRDD[name#0,department#1,salary#2,experience_years#3]



=== Full Execution Plan (extended) ===
== Parsed Logical Plan ==
'Project ['name, 'department, 'salary]
+- Filter (salary#2 > 80000)
   +- LogicalRDD [name#0, department#1, salary#2, experience_years#3], false

== Analyzed Logical Plan ==
name: string, department: string, salary: int
Project [name#0, department#1, salary#2]
+- Filter (salary#2 > 80000)
   +- LogicalRDD [name#0, department#1, salary#2, experience_years#3], false

== Optimized Logical Plan ==
Project [name#0, department#1, salary#2]
+- Filter (isnotnull(salary#2) AND (salary#2 > 80000))
   +- LogicalRDD [name#0, department#1, salary#2, experience_years#3], false

== Physical Plan ==
*(1) Project [name#0, department#1, salary#2]
+- *(1) Filter (isnotnull(salary#2) AND (salary#2 > 80000))
   

---
## 5. RDD vs DataFrame Performance Comparison

A naive benchmark can easily produce a **misleading, reversed result** (RDD appearing faster than DataFrame). Two mistakes cause this:

1. **Setup inside the timer** — putting `createDataFrame(python_list)` in the timed block charges the DataFrame with the cost of serializing every Python row into the JVM. That serialization, not the aggregation, dominates the measurement.
2. **Data too small** — fixed costs (query planning, codegen compilation, shuffle setup) outweigh the actual compute, so neither engine gets to show its strength.

Below we measure three ways so the difference is visible:
- **[A] Flawed** — the common mistake (setup inside timer). DataFrame looks terrible.
- **[B] Fair** — inputs built/cached *outside* the timer, only the aggregation timed, warmed up. DataFrame wins.
- **[C] Native** — data generated inside the JVM via `spark.range` (no Python serialization at all). DataFrame wins by a large margin.

> **The real reason DataFrame wins:** an RDD with Python lambdas runs the function in a **Python worker**, so every row crosses the JVM↔Python boundary (serialize/deserialize). A DataFrame aggregation stays entirely in the **JVM with Tungsten code generation**. Avoiding that boundary — not Catalyst alone — is the dominant effect.

In [14]:
import time

def best_of(fn, repeat=3):
    """Warm up once (JIT, codegen, cache fill), then return the best of N timed runs."""
    fn()
    times = []
    for _ in range(repeat):
        t = time.time(); fn(); times.append(time.time() - t)
    return min(times)

def once(fn):
    t = time.time(); fn(); return time.time() - t

N = 2_000_000
departments = ["Engineering", "Marketing", "Sales", "HR", "Finance"]
py_data = [(f"emp_{i}", departments[i % 5], 50000 + (i % 100000)) for i in range(N)]

# --- Build inputs ONCE, OUTSIDE the timers, and materialize the caches ---
rdd_cached = sc.parallelize(py_data, numSlices=8).cache(); rdd_cached.count()
df_cached  = spark.createDataFrame(py_data, ["name", "department", "salary"]).cache(); df_cached.count()

# === [A] FLAWED: createDataFrame is INSIDE the timed block ===
def flawed_rdd():
    sc.parallelize(py_data).map(lambda x: (x[1], (x[2], 1))) \
      .reduceByKey(lambda a, b: (a[0]+b[0], a[1]+b[1])).mapValues(lambda v: v[0]/v[1]).collect()
def flawed_df():
    d = spark.createDataFrame(py_data, ["name", "department", "salary"])  # <-- setup charged here
    d.groupBy("department").agg(F.avg("salary")).collect()

# === [B] FAIR: prebuilt+cached inputs, only the aggregation is timed ===
def fair_rdd():
    rdd_cached.map(lambda x: (x[1], (x[2], 1))) \
      .reduceByKey(lambda a, b: (a[0]+b[0], a[1]+b[1])).mapValues(lambda v: v[0]/v[1]).collect()
def fair_df():
    df_cached.groupBy("department").agg(F.avg("salary")).collect()

# === [C] NATIVE: data generated in the JVM, no Python serialization ===
gen_df = (spark.range(N)
          .withColumn("department", (F.col("id") % 5).cast("int"))
          .withColumn("salary", 50000 + (F.col("id") % 100000))).cache(); gen_df.count()
gen_rdd = gen_df.rdd.cache(); gen_rdd.count()
def native_rdd():
    gen_rdd.map(lambda r: (r["department"], (r["salary"], 1))) \
      .reduceByKey(lambda a, b: (a[0]+b[0], a[1]+b[1])).mapValues(lambda v: v[0]/v[1]).collect()
def native_df():
    gen_df.groupBy("department").agg(F.avg("salary")).collect()

print(f"N = {N:,} rows\n")
print(f"[A] FLAWED  (createDataFrame inside timer)   RDD: {once(flawed_rdd):6.2f}s   DF: {once(flawed_df):6.2f}s")
print(f"[B] FAIR    (inputs cached, agg only)        RDD: {best_of(fair_rdd):6.3f}s   DF: {best_of(fair_df):6.3f}s")
print(f"[C] NATIVE  (spark.range, no Py serialize)   RDD: {best_of(native_rdd):6.3f}s   DF: {best_of(native_df):6.3f}s")
print("\nTakeaways:")
print("  [A] DataFrame looks slow only because it paid to serialize all Python rows in the timer.")
print("  [B] With a fair setup, DataFrame's JVM/Tungsten aggregation beats the RDD's Python lambdas.")
print("  [C] When no row ever crosses into Python, the DataFrame advantage is largest.")

rdd_cached.unpersist(); df_cached.unpersist(); gen_df.unpersist(); gen_rdd.unpersist()

N = 2,000,000 rows

[A] FLAWED  (createDataFrame inside timer)   RDD:   0.58s   DF:  11.64s
[B] FAIR    (inputs cached, agg only)        RDD:  0.469s   DF:  0.150s
[C] NATIVE  (spark.range, no Py serialize)   RDD:  0.888s   DF:  0.093s

Takeaways:
  [A] DataFrame looks slow only because it paid to serialize all Python rows in the timer.
  [B] With a fair setup, DataFrame's JVM/Tungsten aggregation beats the RDD's Python lambdas.
  [C] When no row ever crosses into Python, the DataFrame advantage is largest.


MapPartitionsRDD[101] at javaToPython at NativeMethodAccessorImpl.java:0

---
## 6. Caching (Persistence)

When the same RDD/DataFrame is used multiple times, **caching** prevents recomputation.

In [15]:
from pyspark import StorageLevel

# Without caching
uncached_df = spark.createDataFrame(py_data, ["name", "department", "salary"]) \
    .filter(F.col("salary") > 80000)

start = time.time()
uncached_df.count()
uncached_df.groupBy("department").avg("salary").collect()
uncached_time = time.time() - start

# With caching
cached_df = spark.createDataFrame(py_data, ["name", "department", "salary"]) \
    .filter(F.col("salary") > 80000) \
    .cache()  # = persist(StorageLevel.MEMORY_ONLY)

# Warm up cache with first action
cached_df.count()

start = time.time()
cached_df.count()
cached_df.groupBy("department").avg("salary").collect()
cached_time = time.time() - start

print(f"Without cache: {uncached_time:.3f}s")
print(f"With cache:    {cached_time:.3f}s")
print(f"\n💡 StorageLevel options:")
print(f"   MEMORY_ONLY      - in-memory only (default)")
print(f"   MEMORY_AND_DISK  - spill to disk when memory is full")
print(f"   DISK_ONLY        - disk only")
print(f"   MEMORY_ONLY_SER  - serialized to reduce memory usage")

# Release cache
cached_df.unpersist()

Without cache: 0.910s
With cache:    0.188s

💡 StorageLevel options:
   MEMORY_ONLY      - in-memory only (default)
   MEMORY_AND_DISK  - spill to disk when memory is full
   DISK_ONLY        - disk only
   MEMORY_ONLY_SER  - serialized to reduce memory usage


DataFrame[name: string, department: string, salary: bigint]

---
## 📝 Key Summary

| Concept | Description |
|------|------|
| **SparkSession** | Entry point to all Spark functionality |
| **RDD** | Low-level distributed data structure; type-safe but not optimizable |
| **DataFrame** | High-level structured data; benefits from Catalyst optimization |
| **Transformation** | Lazy execution; returns new RDD/DataFrame (map, filter, groupBy…) |
| **Action** | Eager execution; returns a value (collect, count, show…) |
| **Lazy Evaluation** | Defers execution until an Action is called → enables optimization |
| **Cache/Persist** | Stores reused data in memory or on disk |

### Next Step (Step 2)
- Spark SQL deep dive & Catalyst Optimizer internals
- Reading execution plans
- Optimization techniques: Predicate pushdown, Column pruning, etc.

In [16]:
# Stop session
spark.stop()
print("SparkSession stopped")

SparkSession stopped
